In [32]:
#import libraries and variables

import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
import joblib
import os

X_train = pd.read_csv("../data/results/X_train.csv")
X_test = pd.read_csv("../data/results/X_test.csv")
Y_train = pd.read_csv("../data/results/Y_train.csv")["Exam_Score"]
Y_test = pd.read_csv("../data/results/Y_test.csv")["Exam_Score"]


In [33]:
# Linear Regression grid:

lr = LinearRegression()

lr_grid = {
    "fit_intercept" : [True, False],
    "positive" : [True, False]
}
lr_search = GridSearchCV(lr, lr_grid, cv=3, scoring="r2", n_jobs=-1)

lr_search.fit(X_train, Y_train)

best_lr = lr_search.best_estimator_

print("best params:", lr_search.best_params_)
print("Best CV R²:", lr_search.best_score_)



best params: {'fit_intercept': True, 'positive': False}
Best CV R²: 0.9898054550471498


##  SVR grid — same X_train/Y_train

In [34]:
svr = SVR()
svr_grid = {
    "C": [1, 10],
    "kernel": ["rbf"],
    "epsilon" : [0.1, 0.5],
}
svr_search = GridSearchCV(svr, svr_grid, cv=3, scoring="r2", n_jobs=-1)
svr_search.fit(X_train, Y_train)

best_svr = svr_search.best_estimator_

print("SVR best params: ", svr_search.best_params_)
print("SVR best CV R²: ", svr_search.best_score_)

SVR best params:  {'C': 10, 'epsilon': 0.1, 'kernel': 'rbf'}
SVR best CV R²:  0.9814007664264603


## compare on the held-out test set (touch it only once):

In [35]:
for name, search in [("LinearRegression", lr_search), ("SVR", svr_search)]:
    y_pred = search.predict(X_test)
    print(f"{name}:  R²={r2_score(Y_test, y_pred):.4f}  MAE={mean_absolute_error(Y_test, y_pred):.4f}")

LinearRegression:  R²=0.9897  MAE=0.2717
SVR:  R²=0.9827  MAE=0.3362


## RandomForest grid

In [36]:
rf = RandomForestRegressor(random_state=42, n_jobs=-1)

rf_grid = {
    "n_estimators" : [100, 200, 300],
    "max_depth" : [None, 10, 20],
    "min_samples_split" : [2, 5, 10]
}

rf_search = GridSearchCV(
    rf,
    rf_grid,
    cv=3,
    scoring="r2",
    n_jobs=-1
)

rf_search.fit(X_train, Y_train)

best_rf = rf_search.best_estimator_

print("RF best params:", rf_search.best_params_)
print("RF best CV R²:", rf_search.best_score_)


RF best params: {'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 300}
RF best CV R²: 0.8823615122663585


## XGBoost grid — same X_train/Y_train

In [37]:
xgb = XGBRegressor(
    random_state=42,
    n_jobs=-1
)

xgb_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.1, 0.2],
    "max_depth": [3, 6, 10],
    "subsample": [0.8, 1.0]
}

xgb_search = GridSearchCV(
    xgb,
    xgb_grid,
    cv=3,
    scoring="r2",
    n_jobs=-1
)

xgb_search.fit(X_train, Y_train)

best_xgb = xgb_search.best_estimator_

print("XGB best params:", xgb_search.best_params_)
print("XGB best CV R²:", xgb_search.best_score_)


XGB best params: {'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 300, 'subsample': 0.8}
XGB best CV R²: 0.9791945417722067


## one-time test comparison

In [38]:

for name, search in [("RF", rf_search), ("XGB", xgb_search)]:
    y_pred = search.predict(X_test)

    print(
        f"{name}: "
        f"R²={r2_score(Y_test, y_pred):.4f}  "
        f"MAE={mean_absolute_error(Y_test, y_pred):.4f}"
    )

RF: R²=0.8927  MAE=0.8398
XGB: R²=0.9821  MAE=0.3425


## evaluate best_estimator_ on the test set

In [39]:
models = [
    ("LinearRegression", best_lr),
    ("SVR", best_svr),
    ("RF", best_rf),
    ("XGB", best_xgb)
]

for name, model in models:
    y_pred = model.predict(X_test)

    print(
        f"{name}: "
        f"R²={r2_score(Y_test, y_pred):.4f}  "
        f"MAE={mean_absolute_error(Y_test, y_pred):.4f}"
    )

LinearRegression: R²=0.9897  MAE=0.2717
SVR: R²=0.9827  MAE=0.3362
RF: R²=0.8927  MAE=0.8398
XGB: R²=0.9821  MAE=0.3425


In [40]:
lr_cv_std = lr_search.cv_results_["std_test_score"][lr_search.best_index_]
svr_cv_std = svr_search.cv_results_["std_test_score"][svr_search.best_index_]
rf_cv_std = rf_search.cv_results_["std_test_score"][rf_search.best_index_]
xgb_cv_std = xgb_search.cv_results_["std_test_score"][xgb_search.best_index_]

lr_cv_mean = lr_search.cv_results_["mean_test_score"][lr_search.best_index_]
svr_cv_mean = svr_search.cv_results_["mean_test_score"][svr_search.best_index_]
rf_cv_mean = rf_search.cv_results_["mean_test_score"][rf_search.best_index_]
xgb_cv_mean = xgb_search.cv_results_["mean_test_score"][xgb_search.best_index_]

cv_std = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "SVR",
        "Random Forest",
        "XGBoost"
    ],
    "CV R² Mean": [
        lr_cv_mean,
        svr_cv_mean,
        rf_cv_mean,
        xgb_cv_mean
    ],
    "CV R² Std": [
        lr_cv_std,
        svr_cv_std,
        rf_cv_std,
        xgb_cv_std
    ]
})

cv_std.to_csv(
    "../data/results/cv_std.csv",
    index=False
)

## before/after table

In [41]:
baseline = pd.read_csv("../data/results/baseline.csv")

print(baseline)

models = [
    ("Linear Regression", best_lr),
    ("SVR", best_svr),
    ("Random Forest", best_rf),
    ("XGBoost", best_xgb)
]

after_results = []

for name, model in models:
    y_pred = model.predict(X_test)

    after_results.append({
        "Model": name,
        "After R²": r2_score(Y_test, y_pred),
        "After MAE": mean_absolute_error(Y_test, y_pred)
    })

after = pd.DataFrame(after_results)


               Model        R²       MAE      RMSE
0  Linear Regression  0.989717  0.271763  0.327151
1      Random Forest  0.891673  0.840500  1.061824
2            XGBoost  0.956514  0.531180  0.672759
3                SVR  0.971842  0.403659  0.541355


## save predictions

In [42]:
predictions = pd.DataFrame({
    "Actual": Y_test,
    "Linear Regression": best_lr.predict(X_test),
    "SVR": best_svr.predict(X_test),
    "Random Forest": best_rf.predict(X_test),
    "XGBoost": best_xgb.predict(X_test)
})

predictions.to_csv(
    "../data/results/predictions.csv",
    index=False
)


## save the best model

In [43]:
os.makedirs("../models", exist_ok=True)

pipeline = Pipeline(["model" , best_lr])
joblib.dump(pipeline, "../models/linear_regression.joblib")
joblib.dump(X_train.columns.tolist(), "../models/columns.joblib")

['../models/columns.joblib']